## Dreambooth LoRA Z-Image

In [ ]:
import gc
import json

import torch
from diffusers import (
    BitsAndBytesConfig,
    ZImagePipeline,
    ZImageTransformer2DModel,
)
from IPython.display import display
from transformers import (
    Qwen2Tokenizer,
    Qwen3Model,
)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
bnb_quantization_config_path = "configs/train_dreambooth_lora_z_image.json"

In [ ]:
try:
	with open(bnb_quantization_config_path, "r") as f:
		config_kwargs = json.load(f)
		if "load_in_4bit" in config_kwargs and config_kwargs["load_in_4bit"]:
			config_kwargs["bnb_4bit_compute_dtype"] = torch.bfloat16
	quantization_config = BitsAndBytesConfig(**config_kwargs)
	for k, v in config_kwargs.items():
		print(f"{k}: {v}")
except:
	pass

### Dog Toy Example

In [ ]:
model_id = "Tongyi-MAI/Z-Image"

In [ ]:
tokenizer = Qwen2Tokenizer.from_pretrained(model_id, subfolder="tokenizer")
text_encoder = Qwen3Model.from_pretrained(
    model_id,
    subfolder="text_encoder",
    torch_dtype=torch.bfloat16,
)

In [ ]:
text_pipe = ZImagePipeline.from_pretrained(
    model_id,
    vae=None,
    transformer=None,
    scheduler=None,
    tokenizer=tokenizer,
    text_encoder=text_encoder,
)

In [ ]:
prompt = "a photo of sks dog in a bucket"
negative_prompt = ""

In [ ]:
with torch.inference_mode():
    prompt_embeds, negative_prompt_embeds = text_pipe.encode_prompt(
        prompt=prompt,
        negative_prompt=negative_prompt,
        max_sequence_length=512,
    )

In [ ]:
del text_pipe
del tokenizer
del text_encoder

gc.collect()
torch.cuda.empty_cache()

In [ ]:
transformer = ZImageTransformer2DModel.from_pretrained(
    model_id,
    subfolder="transformer",
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
)

pipe = ZImagePipeline.from_pretrained(
    model_id,
    tokenizer=None,
    text_encoder=None,
    transformer=transformer,
    torch_dtype=torch.bfloat16,
)
pipe.load_lora_weights("trained-z-image-lora")

In [ ]:
with torch.inference_mode():
	latents = pipe(
		prompt_embeds=prompt_embeds,
		negative_prompt_embeds=negative_prompt_embeds,
		height=1024, width=1024,
		num_inference_steps=50,
		guidance_scale=5.0,
		generator=torch.Generator("cuda").manual_seed(42),
		output_type="latent",
	).images

In [ ]:
del transformer

gc.collect()
torch.cuda.empty_cache()

In [ ]:
latents = (latents / pipe.vae.config.scaling_factor) + pipe.vae.config.shift_factor
latents = latents.to(device="cpu", dtype=torch.bfloat16)
with torch.inference_mode():
    image = pipe.vae.decode(latents, return_dict=False)[0]
image = pipe.image_processor.postprocess(image, output_type="pil")[0]

In [ ]:
display(image)